# Transfer Learning — Classificação Multi-classe de Raio-X
### DenseNet-121 (TorchXRayVision) adaptado para dataset angolano
**Correcções aplicadas:** augmentation, split estratificado, class weighting, map_location, seeds completos, classes.json**

## 1. Instalação e Imports

In [ ]:
# !pip install torchxrayvision scikit-learn matplotlib tqdm

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchxrayvision as xrv
import matplotlib.pyplot as plt
import sklearn.metrics
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# CORRECÇÃO 4 — Reprodutibilidade completa
# PROBLEMA ORIGINAL: só tinha torch.manual_seed(RANDOM_SEED) nas células seguintes.
# Python random, numpy e CUDA também precisam de seed para resultados repetíveis.
# Sem isto, duas execuções do mesmo notebook podiam dar resultados diferentes.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)                          # ← NOVO: seed do Python nativo
np.random.seed(RANDOM_SEED)                       # ← NOVO: seed do numpy
torch.manual_seed(RANDOM_SEED)                    # já existia
torch.cuda.manual_seed_all(RANDOM_SEED)           # ← NOVO: seed do CUDA (GPU)
torch.backends.cudnn.deterministic = True         # ← NOVO: operações CUDA determinísticas
torch.backends.cudnn.benchmark     = False        # ← NOVO: desativa optimização aleatória

print("PyTorch:", torch.__version__)
print("TorchXRayVision:", xrv.__version__)
print("GPU disponível:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

## 2. Configuração — Define aqui as tuas classes e caminhos

In [ ]:
# ════════════════════════════════════════════════════════════
# CONFIGURA AQUI O TEU PROJECTO
# ════════════════════════════════════════════════════════════

CLASSES = [
    "Normal",
    "Pneumonia",
    "Tuberculose",
    "COVID-19"
]
NUM_CLASSES = len(CLASSES)

# Estrutura esperada:
# dataset/
#   Normal/        ← imagens normais
#   Pneumonia/     ← imagens de pneumonia
#   Tuberculose/   ← imagens de TB
#   COVID-19/      ← imagens de COVID
DATASET_PATH = "./dataset"

BATCH_SIZE    = 8
NUM_EPOCHS    = 30
LEARNING_RATE = 1e-4
VAL_SPLIT     = 0.2
TEST_SPLIT    = 0.1

print(f"Classes: {CLASSES}")
print(f"Número de classes: {NUM_CLASSES}")

## 3. Dataset Personalizado

In [ ]:
class XRayDataset(Dataset):
    """
    Dataset personalizado para raio-X com estrutura de pastas por classe.
    Espera:
        dataset/
            Normal/      imagem1.jpg ...
            Pneumonia/   imagem1.jpg ...
    """
    def __init__(self, dataset_path, classes, transform=None):
        self.transform = transform
        self.classes   = classes
        self.samples   = []

        for label_idx, classe in enumerate(classes):
            pasta = os.path.join(dataset_path, classe)
            if not os.path.exists(pasta):
                print(f"Pasta não encontrada: {pasta}")
                continue
            extensoes = (".jpg", ".jpeg", ".png")
            imagens = [f for f in os.listdir(pasta) if f.lower().endswith(extensoes)]
            for img_name in imagens:
                self.samples.append((os.path.join(pasta, img_name), label_idx))
            print(f"  {classe}: {len(imagens)} imagens")

        print(f"\nTotal: {len(self.samples)} imagens")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("L")
        img = np.array(img, dtype=np.float32)
        img = xrv.datasets.normalize(img, 255)
        img = img[None, ...]  # (1, H, W)
        if self.transform:
            img = self.transform(img)
        img = torch.from_numpy(img)
        return img, label

## 4. Carregar Dataset, Augmentation e Split Estratificado

In [ ]:
# CORRECÇÃO 1 — Data Augmentation separado para treino e validação
# PROBLEMA ORIGINAL: havia apenas um bloco de transforms aplicado a todos os conjuntos.
# Com datasets pequenos (caso angolano), o modelo tende a decorar as imagens de treino
# em vez de generalizar — o augmentation cria variações artificiais que simulam
# diferentes condições de captura (posição do paciente, contraste do equipamento).
# IMPORTANTE: o augmentation aplica-se APENAS ao treino; validação e teste
# usam sempre transforms fixos para a avaliação ser justa e comparável.
train_transforms = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224),
    torchvision.transforms.RandomHorizontalFlip(p=0.5),       # ← NOVO: espelha horizontalmente
    torchvision.transforms.RandomRotation(degrees=10),        # ← NOVO: rotação aleatória ±10°
    torchvision.transforms.ColorJitter(brightness=0.2,        # ← NOVO: variação de brilho
                                       contrast=0.2),         #         e contraste (simula diferentes equipamentos)
])

val_test_transforms = torchvision.transforms.Compose([        # ← NOVO: transforms fixos para val/teste
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224)
])
# Nota: o dataset base é carregado SEM transform; o transform é aplicado por subset abaixo


print("A carregar dataset...")
dataset_completo = XRayDataset(DATASET_PATH, CLASSES, transform=None)


# CORRECÇÃO 2 — Split estratificado com sklearn
# PROBLEMA ORIGINAL: torch.utils.data.random_split faz divisão ALEATÓRIA sem garantias
# de distribuição por classe. Com datasets pequenos (ex: 150 imagens, 4 classes),
# é possível que uma classe fique com 0 amostras na validação — o modelo nunca
# aprenderia a detectar essa doença. O split estratificado garante que cada
# classe tem representação proporcional em treino, validação e teste.
indices = list(range(len(dataset_completo)))
labels  = [dataset_completo.samples[i][1] for i in indices]

# Passo 1: separar o conjunto de teste
train_val_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SPLIT,
    stratify=labels,                    # ← garante proporção por classe
    random_state=RANDOM_SEED
)

# Passo 2: dividir treino/validação a partir do que sobrou
train_val_labels = [labels[i] for i in train_val_idx]
val_ratio = VAL_SPLIT / (1.0 - TEST_SPLIT)  # ajustar proporção relativa
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=val_ratio,
    stratify=train_val_labels,          # ← estratificado também aqui
    random_state=RANDOM_SEED
)

# Criar subsets com transforms correctos (treino com aug, val/teste sem)
class SubsetWithTransform(Dataset):
    """Wrapper para aplicar transforms diferentes a subsets do mesmo dataset."""
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform:
            img = self.transform(img.numpy())   # transform espera numpy
            img = torch.from_numpy(img)
        return img, label

train_dataset = SubsetWithTransform(dataset_completo, train_idx, train_transforms)
val_dataset   = SubsetWithTransform(dataset_completo, val_idx,   val_test_transforms)
test_dataset  = SubsetWithTransform(dataset_completo, test_idx,  val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDivisão estratificada do dataset:")
print(f"  Treino:    {len(train_dataset)} imagens")
print(f"  Validação: {len(val_dataset)} imagens")
print(f"  Teste:     {len(test_dataset)} imagens")


# CORRECÇÃO 3 — Class weighting para dados desequilibrados
# PROBLEMA ORIGINAL: o modelo tratava todas as classes igualmente.
# Se tiveres 200 imagens de Pneumonia e 50 de COVID-19, o modelo aprende
# a ignorar COVID-19 (dá sempre Pneumonia e fica com 80% de accuracy).
# O class weighting penaliza mais os erros nas classes raras, forçando
# o modelo a aprender todas as classes. O peso é inversamente proporcional
# à frequência: classes raras têm peso maior.
contagens    = Counter(label for _, label in dataset_completo.samples)
total_imgs   = sum(contagens.values())
class_weights = torch.tensor(
    [total_imgs / (NUM_CLASSES * contagens.get(i, 1)) for i in range(NUM_CLASSES)],
    dtype=torch.float
).to(device)
print(f"\nPesos por classe (class weighting):")
for i, (cls, w) in enumerate(zip(CLASSES, class_weights)):
    print(f"  {cls}: {contagens.get(i,0)} imgs → peso {w:.3f}")


# CORRECÇÃO 5 — Guardar o mapeamento de classes junto com o modelo
# PROBLEMA ORIGINAL: o ficheiro .pth guardava apenas os pesos da rede.
# Ao carregar o modelo no FastAPI (Python), não havia forma de saber
# que índice 0 = "Normal", índice 1 = "Pneumonia", etc.
# Este ficheiro JSON resolve esse problema — carrega-o sempre com o modelo.
with open("classes.json", "w", encoding="utf-8") as f:
    json.dump({"classes": CLASSES, "num_classes": NUM_CLASSES}, f, ensure_ascii=False, indent=2)
print("\nclasses.json guardado — necessário para carregar o modelo no FastAPI")

## 5. Modelo — DenseNet-121 com Transfer Learning

In [ ]:
print("A carregar modelo pré-treinado...")
# Usar o MESMO peso do modelo_base_inferencia.ipynb para consistência
model = xrv.models.DenseNet(weights="densenet121-res224-mimic_ch")
model.op_threshs = None  # desativar calibração original do modelo base

# ── TRANSFER LEARNING — Fase 1: congelar backbone, treinar só o classifier ──
for param in model.parameters():
    param.requires_grad = False

# Substituir o classifier por um novo para NUM_CLASSES classes
# 1024 = número de features do DenseNet-121 (arquitectura fixa, não alterar)
model.classifier = nn.Sequential(
    nn.Linear(1024, 256),
    nn.ReLU(),
    nn.Dropout(0.4),        # regularização — importante com poucos dados
    nn.Linear(256, NUM_CLASSES)
)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros totais:     {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")
print(f"Parâmetros congelados: {total_params - trainable_params:,}")
print(f"\nModelo pronto — só o classifier vai ser treinado na Fase 1.")

## 6. Treino — Fase 1 (só o classifier)

In [ ]:
# CORRECÇÃO 3 (continuação) — CrossEntropyLoss com class_weights
# PROBLEMA ORIGINAL: nn.CrossEntropyLoss() sem pesos tratava todas as classes igual.
# Agora passa os pesos calculados na célula de configuração do dataset.
criterion = nn.CrossEntropyLoss(weight=class_weights)   # ← CORRECÇÃO: adiciona weight=

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)


# ── Funções auxiliares de treino e validação ──────────────────────────────────
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(loader, desc="Treino", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100. * correct / total

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Validação", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), 100. * correct / total


# ── Loop de treino principal ──────────────────────────────────────────────────
historico = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
melhor_val_loss       = float("inf")
epochs_sem_melhoria   = 0
PACIENCIA_EARLY_STOP  = 10

print("A iniciar treino (Fase 1 — só classifier)...\n")
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = val_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)

    historico["train_loss"].append(train_loss)
    historico["val_loss"].append(val_loss)
    historico["train_acc"].append(train_acc)
    historico["val_acc"].append(val_acc)

    print(f"Época {epoch+1:02d}/{NUM_EPOCHS} | "
          f"Loss treino: {train_loss:.4f} | Acc treino: {train_acc:.1f}% | "
          f"Loss val: {val_loss:.4f} | Acc val: {val_acc:.1f}%")

    if val_loss < melhor_val_loss:
        melhor_val_loss = val_loss
        torch.save(model.state_dict(), "melhor_modelo.pth")
        print(f"  Melhor modelo guardado (val_loss: {val_loss:.4f})")
        epochs_sem_melhoria = 0
    else:
        epochs_sem_melhoria += 1
        if epochs_sem_melhoria >= PACIENCIA_EARLY_STOP:
            print(f"\nEarly stopping na época {epoch+1}")
            break

print("\nFase 1 concluída!")

## 7. Fine-tuning — Fase 2 (descongelar últimas camadas)

In [ ]:
# CORRECÇÃO (map_location) — Carregar modelo sem depender de GPU
# PROBLEMA ORIGINAL: torch.load("melhor_modelo.pth") sem map_location.
# Se o treino foi feito em GPU (CUDA) e depois tentares carregar o modelo
# num computador sem GPU (ex: servidor FastAPI sem GPU), o PyTorch lança:
#   RuntimeError: Attempting to deserialize object on a CUDA device but...
# Com map_location=device, o PyTorch carrega os pesos no dispositivo correcto
# automaticamente, seja CPU ou GPU.
model.load_state_dict(torch.load("melhor_modelo.pth", map_location=device))  # ← CORRECÇÃO: adiciona map_location

# Descongelar o último bloco denso do DenseNet-121 para fine-tuning
for param in model.features.denseblock4.parameters():
    param.requires_grad = True
for param in model.features.norm5.parameters():
    param.requires_grad = True

# Learning rate MUITO menor para não destruir o pré-treino do backbone
optimizer_ft = torch.optim.Adam([
    {"params": model.classifier.parameters(),            "lr": 1e-4},
    {"params": model.features.denseblock4.parameters(), "lr": 1e-5},
    {"params": model.features.norm5.parameters(),       "lr": 1e-5},
])
scheduler_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ft, mode='min', factor=0.5, patience=5, verbose=True
)

trainable_ft = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis na Fase 2: {trainable_ft:,}")

NUM_EPOCHS_FT          = 15
melhor_val_loss_ft     = float("inf")
epochs_sem_melhoria_ft = 0

print("\nA iniciar Fine-tuning (Fase 2)...\n")
for epoch in range(NUM_EPOCHS_FT):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_ft)
    val_loss,   val_acc   = val_epoch(model, val_loader, criterion)
    scheduler_ft.step(val_loss)

    print(f"Época {epoch+1:02d}/{NUM_EPOCHS_FT} | "
          f"Loss treino: {train_loss:.4f} | Acc treino: {train_acc:.1f}% | "
          f"Loss val: {val_loss:.4f} | Acc val: {val_acc:.1f}%")

    if val_loss < melhor_val_loss_ft:
        melhor_val_loss_ft = val_loss
        torch.save(model.state_dict(), "melhor_modelo_finetuned.pth")
        print(f"  Melhor modelo fine-tuned guardado (val_loss: {val_loss:.4f})")
        epochs_sem_melhoria_ft = 0
    else:
        epochs_sem_melhoria_ft += 1
        if epochs_sem_melhoria_ft >= 8:
            print(f"\nEarly stopping na época {epoch+1}")
            break

print("\nFine-tuning concluído!")

## 8. Gráficos de treino

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(historico["train_loss"], label="Treino",    color="#378ADD")
ax1.plot(historico["val_loss"],   label="Validação", color="#D85A30")
ax1.set_title("Loss por época")
ax1.set_xlabel("Época")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(historico["train_acc"], label="Treino",    color="#378ADD")
ax2.plot(historico["val_acc"],   label="Validação", color="#D85A30")
ax2.set_title("Accuracy por época")
ax2.set_xlabel("Época")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("graficos_treino.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráficos guardados em graficos_treino.png")

## 9. Avaliação final no conjunto de teste

In [ ]:
# CORRECÇÃO (map_location) — mesma correcção aplicada ao modelo final
# Sem map_location, falha ao carregar em CPU um modelo treinado em GPU.
model.load_state_dict(torch.load("melhor_modelo_finetuned.pth", map_location=device))  # ← CORRECÇÃO
model.eval()

all_labels = []
all_preds  = []
all_probs  = []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="A avaliar"):
        imgs    = imgs.to(device)
        outputs = model(imgs)
        probs   = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_labels.extend(labels.numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

accuracy = sklearn.metrics.accuracy_score(all_labels, all_preds)
print(f"\n{'='*50}")
print(f"  RESULTADOS FINAIS — CONJUNTO DE TESTE")
print(f"{'='*50}")
print(f"  Accuracy: {accuracy*100:.2f}%")

print(f"\n  F1-score por classe:")
f1_scores = sklearn.metrics.f1_score(all_labels, all_preds, average=None)
for classe, f1 in zip(CLASSES, f1_scores):
    print(f"    {classe}: {f1:.4f}")
print(f"  F1-score médio: {f1_scores.mean():.4f}")

print(f"\n  AUC por classe:")
from sklearn.preprocessing import label_binarize
labels_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
auc_scores = []
for i, classe in enumerate(CLASSES):
    try:
        auc = sklearn.metrics.roc_auc_score(labels_bin[:, i], all_probs[:, i])
        auc_scores.append(auc)
        print(f"    {classe}: {auc:.4f}")
    except Exception:
        print(f"    {classe}: N/A (classe não presente no teste)")
if auc_scores:
    print(f"  AUC médio: {np.mean(auc_scores):.4f}")
print(f"{'='*50}")

## 10. Matriz de Confusão

In [ ]:
import itertools

cm = sklearn.metrics.confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im)

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticklabels(CLASSES)

thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, str(cm[i, j]),
            ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black')

ax.set_ylabel('Classe real')
ax.set_xlabel('Classe prevista')
ax.set_title('Matriz de Confusão — Conjunto de Teste')
plt.tight_layout()
plt.savefig("matriz_confusao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Matriz guardada em matriz_confusao.png")

## 11. Relatório completo de classificação

In [ ]:
report = sklearn.metrics.classification_report(
    all_labels, all_preds,
    target_names=CLASSES,
    digits=4
)
print(report)

with open("relatorio_classificacao.txt", "w") as f:
    f.write(report)
print("Relatório guardado em relatorio_classificacao.txt")

## 12. Testar com uma imagem individual

In [ ]:
def prever_imagem(caminho_imagem, model, classes):
    """
    Faz predição de uma imagem individual e mostra o resultado visualmente.
    Esta função é a base para o endpoint FastAPI — a lógica de pré-processamento
    deve ser idêntica entre aqui e o servidor Python.
    """
    transform = torchvision.transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(224)
    ])

    img = Image.open(caminho_imagem).convert("L")
    img = np.array(img, dtype=np.float32)
    img = xrv.datasets.normalize(img, 255)
    img = img[None, ...]
    img = transform(img)
    img = torch.from_numpy(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(img)
        probs   = torch.softmax(outputs, dim=1)[0].cpu().numpy()
        pred_idx = probs.argmax()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    img_show = Image.open(caminho_imagem).convert("L")
    ax1.imshow(img_show, cmap='gray')
    ax1.set_title(f"Imagem: {os.path.basename(caminho_imagem)}")
    ax1.axis('off')

    cores = ['#378ADD' if i != pred_idx else '#1D9E75' for i in range(len(classes))]
    bars  = ax2.barh(classes, probs * 100, color=cores)
    ax2.set_xlabel('Probabilidade (%)')
    ax2.set_title('Predição por classe')
    ax2.set_xlim(0, 100)
    for bar, prob in zip(bars, probs):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{prob*100:.1f}%', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f"\nDiagnóstico: {classes[pred_idx]} ({probs[pred_idx]*100:.1f}%)")
    return classes[pred_idx], probs


# Exemplo de uso — altera o caminho para a tua imagem angolana
# prever_imagem("caminho/para/imagem.jpg", model, CLASSES)